In [ ]:
from dotenv import load_dotenv
load_dotenv()

from langchain.chat_models import init_chat_model

model = init_chat_model("openai:gpt-5.4-nano", temperature=0)

In [ ]:
from typing import Annotated
from typing_extensions import TypedDict
from langchain.messages import AnyMessage
from langgraph.graph.message import add_messages


class State(TypedDict):
    messages: Annotated[list[AnyMessage], add_messages]
    llm_calls: int

In [ ]:
from langchain.tools import tool


@tool
def add(a: int, b: int) -> int:
    """Add two numbers."""
    return a + b


@tool
def multiply(a: int, b: int) -> int:
    """Multiply two numbers."""
    return a * b
@tool
def divide(a: int, b: int) -> float:
    """Divide `a` and `b`.

    Args:
        a: First int
        b: Second int
    """
    return a / b


tools = [add, multiply, divide]
tools_by_name = {t.name: t for t in tools}
model_with_tools = model.bind_tools(tools)

In [ ]:
from langchain.messages import SystemMessage, ToolMessage


def llm_call(state: State) -> dict:
    """Model decides whether to answer or call a tool."""
    response = model_with_tools.invoke(
        [SystemMessage("You are a helpful assistant that does arithmetic.")]
        + state["messages"]
    )
    return {"messages": [response], "llm_calls": state.get("llm_calls", 0) + 1}






def tool_node(state: State) -> dict:
    """Executes whichever tool(s) the model called."""
    results = []
    for call in state["messages"][-1].tool_calls:
        tool_fn = tools_by_name[call["name"]]
        observation = tool_fn.invoke(call["args"])
        results.append(ToolMessage(content=str(observation), tool_call_id=call["id"]))
    return {"messages": results}

In [ ]:
from typing import Literal
from langgraph.graph import StateGraph, START, END


def should_continue(state: State) -> Literal["tool_node", "__end__"]:
    last_message = state["messages"][-1]
    return "tool_node" if last_message.tool_calls else END

In [ ]:
builder = StateGraph(State)
builder.add_node("llm_call", llm_call)
builder.add_node("tool_node", tool_node)

builder.add_edge(START, "llm_call")
builder.add_conditional_edges("llm_call", should_continue, ["tool_node", END])
builder.add_edge("tool_node", "llm_call")

agent = builder.compile()

In [ ]:
from IPython.display import Image, display

display(
    Image(
        agent.get_graph().draw_mermaid_png()
    )
)

In [ ]:
result = agent.invoke({"messages": [{"role": "user", "content": "what is (30 + 48) * 28470?"}]})
result

In [ ]:
result['messages']

In [ ]:
print(result['messages'][-1].content)

In [ ]:
for m in result["messages"]:
    m.pretty_print()

In [ ]:
print(len(tools))

In [ ]:
print(len(tools_by_name))

In [ ]:
tools

In [ ]:
from langchain.agents import create_agent

prebuilt_agent = create_agent(
    model="openai:gpt-5.4-nano",
    tools=tools,
    system_prompt="You are a helpful assistant that does arithmetic.")
   

In [ ]:
from IPython.display import Image, display

display(
    Image(
        prebuilt_agent.get_graph().draw_mermaid_png()
    )
)

In [ ]:
output=prebuilt_agent.invoke(
    {"messages": [{"role": "user", "content": "what is 9 * 6+44?"}]},
)

In [ ]:
for m in output["messages"]:
    m.pretty_print()